# SCF Cohort GLM — Complete MLOps Walkthrough

This notebook explains every stage of the SCF savings cashflow GLM pipeline: what the code does, what the GLM model does, and what gets stored at each step.

**Model:** Generalised Linear Model (GLM) — predicts monthly balance, receipts, withdrawals, and transfers for savings account cohorts.

**Business purpose:** Project savings balances 12–60 months forward per cohort × product for regulatory stress-testing and customer retention strategy.

**Accuracy:** 2.28% Mean APE on a £9–11bn portfolio (out-of-sample).

## Step 1 — Data Ingestion (Delta time-travel pin)
**File:** `src/training/01_data_ingestion.py` + `src/common/dataset_versioning.py`

Reads 3 source files from a UC Volume into Bronze Delta tables:

| Source file | Delta table | Contains |
|---|---|---|
| `base_query_df_apr_2026.parquet` | `raw_base_data` | Account-level monthly balances, flows, rates |
| `moneyfacts_merge.parquet` | `raw_moneyfacts_best_buy` | Market best-buy rates per product per month |
| `QRM_products_revised.csv` | `raw_qrm_products` | Product code to QRM category mapping |

Every ingestion writes the exact Delta version number to `dataset_versions`, so any training run can be reproduced with `VERSION AS OF`.

## Step 2 — Data Validation (schema + quality gates)
**Files:** `src/training/02b_data_validation.py` + `src/inference/06_data_quality.py`

Six gates checked before the GLM sees any data: row count >= 500, null rate < 5% on balance/int_rate, outflow_prop in [0,1], all 17 schema columns present, >= 5 distinct products. Task fails immediately on any breach.

## Step 3 — Feature Engineering (point-in-time joins)
**Files:** `src/training/02_data_preprocessing.py` + `src/training/00_feature_engineering_redshift.py`

Accounts grouped into cohorts, then features computed: rec_prop, outflow_prop, withdrawal_prop_of_outflow, delta_to_best_buy, and dbb_range (11 bins). The `as_of_date` widget prevents data leakage. RATE_BINS/RATE_LABELS shared between training and inference prevent train/serve skew.

## Step 4 — Feature Store (freshness checks)
**Method:** `FeatureEngineeringClient.write_table(mode="merge")`

Feature table `{catalog}.feature_store.feature_store_cohort` with PRIMARY KEY (cohort, product, months_since_start). Enforces primary key + native UC lineage: raw_data -> features -> model -> predictions. Lakehouse Monitoring tracks feature distribution shift.

## Step 5 — Model Training (the 3 GLMs)
**File:** `src/training/03_model_training.py`

Three GLMs fitted per product (10 products x 3 = 30 GLMs total):
- GLM 1 (Gamma/Tweedie): rec_prop ~ cr(months_since_start, df=4) + delta_to_best_buy
- GLM 2 (Binomial): outflow_prop ~ cr(months_since_start, df=4) + delta_to_best_buy
- GLM 3 (Binomial): withdrawal_prop_of_outflow ~ delta_to_best_buy

Fitted GLMs converted to lookup tables saved to UC Volume as latest_training_run.pkl.

### The autoregressive projection chain
```
receipts_t    = rec_pct[month, bin]     * balance_lag_1
outflow_t     = outflow_pct[month, bin] * (balance_lag_1 + receipts_t)
transfers_t   = outflow_t * transfer_split
withdrawals_t = outflow_t - transfers_t
balance_t     = balance_lag_1 + receipts_t - withdrawals_t - transfers_t
# balance_t becomes balance_lag_1 for month t+1 (chained up to 60 months)
```

## Step 6 — Hyperparameter Tuning (nested MLflow runs)
**File:** `src/training/03_model_training.py`

Grid: month_end in {36,48,50,60} x drop_month2 in {True,False} = 5 configs per product = 50 trials. Each trial is a nested MLflow child run. 3-fold rolling-origin cross-validation per config gives cv_mape. Best config selected by lowest val_mape. Baseline benchmark: GLM must beat naive last-value forecast.

## Step 7 — Model Evaluation (held-out metrics)
**File:** `src/training/04_model_evaluation.py`

Computes mape_balance, baseline_mape, improvement_vs_baseline_pct, beats_baseline per product. portfolio_mape = mean across products (the Champion/Challenger metric). Written to model_eval_metrics for full history.

## Step 8 — Champion vs Challenger (offline comparison)
**Files:** `src/training/05_model_registration.py` + `src/common/mlflow_utils.py`

Challenger promoted to @champion only if MAPE improves by more than 15% (MAPE_PROMOTION_THRESHOLD). Example: champion=5.20%, challenger=4.85%, improvement=6.7% < 15% -> NOT promoted. First-ever run always promotes.

## Step 9 — MLflow Tracking
Every run logs params (cutoff_period, n_products, best HPO config), metrics (portfolio_mape, per-product MAPE, beats_baseline), artifacts (SCFCohortModel pyfunc + signature), and tags (git commit, cluster ID). The system of record for lineage.

## Step 10 — Model Registry (Unity Catalog)
**File:** `src/common/mlflow_utils.py`

{catalog}.ml_models.scf_cohort_model with aliases @baseline (v1, permanent), @challenger (latest), @champion (production). Inference loads @champion, never a version number.

In [ ]:
import mlflow
import pandas as pd
mlflow.set_registry_uri('databricks-uc')

# model = mlflow.pyfunc.load_model('models:/pd_dtl_ds.ml_models.scf_cohort_model@champion')
# predictions = model.predict(pd.DataFrame([{
#     'product': 'EA ISA (Online)', 'cohort': '2023-03',
#     'reporting_period': '2026-07', 'months_since_start': 40,
#     'dbb_range': '[-0.5, -0.4)', 'balance_lag_1': 45230.0
# }]))
print('Inference loads @champion alias - automatic after promotion')

## Step 11 — Model Card Auto-Gen
**File:** `src/training/05_model_registration.py` -> model_cards table

Auto-generated in the same job as registration (can never drift from artifact). Contains model name, version, run_id, features, labels, portfolio MAPE, promotion decision. Primary input the FMC reviewer reads.

## Step 12 — Second-Line Validation (FMC gate)
**Files:** `src/training/05a_fmc_validation_gate.py` + `fmc_approve.py`

Writes PENDING to model_approvals, sends Teams notification, polls every 5 min for 24h. FMC reviewer runs fmc_approve.py with APPROVED/REJECTED. Approved -> registration proceeds. Rejected -> version keeps @challenger, never reaches inference, archived next cycle.

## Step 13 — Model Deployment (zero-downtime, gradual rollout)
**File:** `src/training/07_deploy_serving_endpoint.py`

90% champion / 10% challenger shadow testing via traffic_percentage. Zero-downtime update_config_and_wait(). Every deploy recorded in deployment_history.

## Step 14 — Smoke Tests & Auto-Rollback
**Files:** `src/training/08_smoke_test_serving_endpoint.py` + `09_auto_rollback_guard.py`

Smoke test sends 1 real row and asserts non-empty prediction. If it fails, auto_rollback_guard re-points @champion to prior version, resets endpoint to 100% prior champion, and writes rollback_events. A bad deployment never stays live.

## Step 15 — Notifications
**Files:** `src/common/notifications.py` + confirmation_mail.py

Training complete (email+Teams with MLflow link + MAPE), drift ALERT (PSI per feature), FMC review required (request ID + comparison), auto-rollback (reason + version). Credentials from Databricks secret scope, never in code.

## GLM Model Summary

**Input** (one row per cohort x product x month): product, cohort, reporting_period, months_since_start, dbb_range, balance_lag_1

**Output:** balance_pred, receipts_pred, withdrawals_pred, transfers_pred + APE columns + ratio columns (rec_pct_pred, outflow_pct_pred, transfer_pct_pred)

**Why GLM (not neural network)?** Interpretable coefficients for regulatory explainability, works on small cohort-level data, stable predictions, no GPU needed. The model IS three lookup tables per product that a regulator can read directly.